In [ ]:
!pip install ultralytics supervision opencv-python numpy pandas -q

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict
from google.colab import files
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
import base64

print("✅ Libraries Installed.")

In [ ]:
def draw_zone(image_path, zone_name):
  # Resize to prevent browser crash
  frame = cv2.imread(image_path)
  h, w = frame.shape[:2]
  scale = 800 / w
  resized = cv2.resize(frame, (int(w*scale), int(h*scale)))
  _, buf = cv2.imencode('.jpg', resized)
  img_data = 'data:image/jpeg;base64,' + base64.b64encode(buf).decode('utf-8')

  js = Javascript('''
    async function draw(img_src, zone_name) {
      const div = document.createElement('div');
      div.innerHTML = `<h3>Draw ${zone_name} (Click corners, Double-click to finish)</h3><canvas id="c"></canvas><br><button id="btn">Save & Continue</button>`;
      document.body.append(div);
      const canvas = document.getElementById('c');
      const ctx = canvas.getContext('2d');
      const img = new Image(); img.src = img_src;
      await new Promise(r => img.onload = r);
      canvas.width = img.width; canvas.height = img.height;
      ctx.drawImage(img,0,0);
      let pts = [];
      canvas.onclick = e => {
        const r = canvas.getBoundingClientRect();
        pts.push([Math.round((e.clientX-r.left)), Math.round((e.clientY-r.top))]);
        ctx.fillStyle='lime'; ctx.fillRect(pts[pts.length-1][0]-2, pts[pts.length-1][1]-2, 4, 4);
        if (pts.length>1) {
            ctx.beginPath(); ctx.moveTo(pts[pts.length-2][0], pts[pts.length-2][1]);
            ctx.lineTo(pts[pts.length-1][0], pts[pts.length-1][1]); ctx.strokeStyle='lime'; ctx.stroke();
        }
      };
      canvas.ondblclick = () => {
        ctx.beginPath(); ctx.moveTo(pts[pts.length-1][0], pts[pts.length-1][1]);
        ctx.lineTo(pts[0][0], pts[0][1]); ctx.stroke();
      };
      await new Promise(r => document.getElementById('btn').onclick = () => { div.remove(); r(); });
      return pts;
    }
  ''')
  display(js)
  pts = eval_js(f'draw("{img_data}", "{zone_name}")')
  return (np.array(pts) / scale).astype(int)

# --- EXECUTE DRAWING ---
VIDEO_PATH = "eee.mp4" #file_name 
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
if ret:
    cv2.imwrite("ref.jpg", frame)
    print("👉 Step 1: Draw the FLOOR (Walking Area)")
    FLOOR_POLY = draw_zone("ref.jpg", "FLOOR ZONE")
    
    print("👉 Step 2: Draw SHELF A (First Aisle/Section)")
    SHELF_A_POLY = draw_zone("ref.jpg", "SHELF A")

    print("👉 Step 3: Draw SHELF B (Second Aisle/Section) - Double Click any spot if none.")
    SHELF_B_POLY = draw_zone("ref.jpg", "SHELF B")
    
    print("✅ Zones Saved.")
else:
    print("❌ Error: Upload 'decathlon_sample.mp4' first.")

In [ ]:
# --- CONFIGURATION ---
OUTPUT_VIDEO = "final_output_stats.mp4"
OUTPUT_CSV = "store_analytics.csv"
WRIST_CROP = 140
CONFIDENCE_REQ = 2 # Frames to confirm product

# Relevant Object IDs (COCO) - filtering out people/ties/chairs
VALID_ITEMS = [24, 25, 26, 28, 39, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 64, 65, 66, 67, 73, 74, 75, 76, 77]

class Customer:
    def __init__(self, id):
        self.id = id
        self.entry = 0
        self.exit = 0
        self.status = "WALKING" # WALKING, BROWSING, TAKEN
        self.product = None
        self.zone = "FLOOR"
        self.buffer = 0

def main():
    # Load Models (Small variants for balance of speed/accuracy)
    print("⏳ Loading Models...")
    pose_model = YOLO('yolov8l-pose.pt') 
    obj_model = YOLO('yolov8l.pt')

    cap = cv2.VideoCapture(VIDEO_PATH)
    W, H = int(cap.get(3)), int(cap.get(4))
    fps = int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    # Zones
    zones = {
        "SHELF_A": sv.PolygonZone(SHELF_A_POLY, triggering_anchors=[sv.Position.CENTER]),
        "SHELF_B": sv.PolygonZone(SHELF_B_POLY, triggering_anchors=[sv.Position.CENTER])
    }
    floor_poly = sv.PolygonZone(FLOOR_POLY)

    customers = {}
    frame_n = 0
    print(f"🚀 Starting Analysis ({W}x{H})...")

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        frame_n += 1
        curr_time = frame_n / fps
        
        # 1. Pose Tracking
        results = pose_model.track(frame, persist=True, verbose=False)
        
        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.int().cpu().numpy()
            boxes = results[0].boxes.xyxy.cpu().numpy()
            kpts = results[0].keypoints.xy.cpu().numpy()

            for i, tid in enumerate(ids):
                if tid not in customers: 
                    customers[tid] = Customer(tid)
                    customers[tid].entry = curr_time
                
                c = customers[tid]
                c.exit = curr_time
                box = boxes[i]
                
                # Default Visuals
                color = (0, 255, 0) # Green (Innocent)
                
                # 2. Check Shelf Interaction
                # Keypoints: 9=LeftW, 10=RightW
                lw, rw = kpts[i][9], kpts[i][10]
                
                active_shelf = None
                wrist_pt = None
                
                # Check Shelf A
                if cv2.pointPolygonTest(SHELF_A_POLY, (lw[0],lw[1]), False) >= 0: wrist_pt, active_shelf = lw, "SHELF_A"
                elif cv2.pointPolygonTest(SHELF_A_POLY, (rw[0],rw[1]), False) >= 0: wrist_pt, active_shelf = rw, "SHELF_A"
                # Check Shelf B
                elif cv2.pointPolygonTest(SHELF_B_POLY, (lw[0],lw[1]), False) >= 0: wrist_pt, active_shelf = lw, "SHELF_B"
                elif cv2.pointPolygonTest(SHELF_B_POLY, (rw[0],rw[1]), False) >= 0: wrist_pt, active_shelf = rw, "SHELF_B"

                # 3. Logic
                if c.status == "TAKEN":
                    color = (0, 0, 255) # Stay Red
                elif active_shelf:
                    c.status = "BROWSING"
                    color = (0, 255, 255) # Yellow
                    c.zone = active_shelf
                    
                    # TRIGGER PRODUCT DETECTION (Every 3 frames to save speed)
                    if frame_n % 3 == 0:
                        x, y = int(wrist_pt[0]), int(wrist_pt[1])
                        x1, x2 = max(0, x-WRIST_CROP), min(W, x+WRIST_CROP)
                        y1, y2 = max(0, y-WRIST_CROP), min(H, y+WRIST_CROP)
                        crop = frame[y1:y2, x1:x2]
                        
                        if crop.size > 0:
                            # Detect
                            res = obj_model(crop, verbose=False, conf=0.15)
                            for b in res[0].boxes:
                                if int(b.cls[0]) in VALID_ITEMS:
                                    c.buffer += 1
                                    # Draw Crop Box
                                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
                                    if c.buffer >= CONFIDENCE_REQ:
                                        c.status = "TAKEN"
                                        c.product = res[0].names[int(b.cls[0])]
                                        break
                else:
                    if c.status != "TAKEN": c.status = "WALKING"

                # 4. Draw Stats on Person
                label = f"ID:{tid} {c.status}"
                if c.product: label += f" [{c.product}]"
                
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)
                cv2.putText(frame, label, (int(box[0]), int(box[1])-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                
                # Draw Skeleton (Arms only for cleanliness)
                for pt in [kpts[i][5], kpts[i][6], kpts[i][7], kpts[i][8], kpts[i][9], kpts[i][10]]:
                    if pt[0] > 0: cv2.circle(frame, (int(pt[0]), int(pt[1])), 4, (255,200,0), -1)

        # Draw Zones
        cv2.polylines(frame, [FLOOR_POLY], True, (50, 50, 50), 1)
        cv2.polylines(frame, [SHELF_A_POLY, SHELF_B_POLY], True, (255, 255, 0), 2)
        
        # Dashboard Stats
        cv2.rectangle(frame, (0,0), (300, 40), (0,0,0), -1)
        cv2.putText(frame, f"People: {len(customers)} | Active: {len(ids) if 'ids' in locals() else 0}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        out.write(frame)
        if frame_n % 10 == 0: print(f"Processing {curr_time:.1f}s...", end='\r')

    cap.release()
    out.release()

    # Generate Report
    rows = []
    for c in customers.values():
        rows.append({
            "Customer_ID": c.id,
            "Zone_Interaction": c.zone,
            "Status": c.status,
            "Product_Taken": c.product if c.product else "None",
            "Dwell_Time_Sec": round(c.exit - c.entry, 2)
        })
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    
    print("\n✅ DONE. Downloading...")
    files.download(OUTPUT_VIDEO)
    files.download(OUTPUT_CSV)

if __name__ == "__main__":
    if 'SHELF_A_POLY' in globals(): main()
    else: print("❌ Run Step 2 (Drawing) first!")